
# Лабораторная работа №3

## Тема: Двойственность и анализ чувствительности в задаче распределения публичного бюджета

> В этой работе двойственность используется как рабочий инструмент анализа, а не как отдельная формальная цель.
> Фокус: понять, какие ресурсы реально дефицитны, проверить сильную двойственность и сравнить прогноз по shadow price с фактическим пересчётом модели.

### Как читать эту работу

- $x_j$ означает масштаб запуска программы $j$ от `0` до `1`.
- Ограничения показывают, сколько ресурса тратится на каждую программу.
- `binding` означает, что ресурс исчерпан в оптимуме.
- `slack` означает, что часть ресурса осталась в резерве.
- Теневая цена ресурса показывает, насколько ценна ещё одна единица этого ресурса в текущем оптимуме.



## 1. Цель работы

Освоить полный цикл sensitivity-first анализа на одном и том же примере:

1. собрать прямую модель распределения публичного бюджета;
2. решить её через `scipy.optimize.linprog`;
3. определить активные и неактивные ограничения;
4. кратко выписать и численно проверить двойственную модель;
5. провести эксперименты по изменению правых частей и коэффициентов цели.

## 2. Формируемые умения и навыки

После выполнения работы вы сможете:

1. отличать `binding`-ограничения от ограничений с запасом;
2. интерпретировать теневые цены ресурсов без длинного ручного симплекс-анализа;
3. проверять сильную двойственность численно;
4. оценивать эффект малых изменений ресурсов по shadow price;
5. понимать, когда изменение параметров меняет только значение оптимума, а когда меняет сам состав оптимального плана.



## 3. Исходный кейс

Регион распределяет ограниченные ресурсы между четырьмя программами.

Переменные:

- $x_1$ — доля запуска программы **мобильные медицинские бригады**;
- $x_2$ — доля запуска программы **школьное питание**;
- $x_3$ — доля запуска программы **цифровые образовательные наборы**;
- $x_4$ — доля запуска программы **зимние центры поддержки населения**.

Для каждой программы разрешён диапазон:

$$
0 \le x_j \le 1.
$$

Ресурсы:

1. бюджет, млн руб.;
2. трудозатраты персонала, условные командо-месяцы;
3. операционная ёмкость, условные организационные слоты.

Целевая функция измеряет суммарный общественный эффект в условных баллах.


In [1]:

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import linprog

programs = [
    'Мобильные медбригады',
    'Школьное питание',
    'Цифровые наборы',
    'Зимние центры поддержки',
]
resources = [
    'Бюджет, млн руб.',
    'Трудозатраты, командо-месяцы',
    'Операционная ёмкость, слоты',
]

effect = np.array([90, 78, 70, 96], dtype=float)
A_resources = np.array([
    [40, 28, 22, 48],
    [20, 24, 12, 26],
    [16, 12, 10, 20],
], dtype=float)
b_resources = np.array([88, 52, 42], dtype=float)
bounds = [(0, 1)] * len(programs)

profile_df = pd.DataFrame({
    'Программа': programs,
    'Эффект при полном масштабе': effect,
    'Бюджет, млн руб.': A_resources[0],
    'Трудозатраты': A_resources[1],
    'Операционная ёмкость': A_resources[2],
})
limits_df = pd.DataFrame({
    'Ресурс': resources,
    'Доступный лимит': b_resources,
})

display(profile_df)
display(limits_df)


,Программа,Эффект при полном масштабе,"Бюджет, млн руб.",Трудозатраты,Операционная ёмкость
0,Мобильные медбригады,90.0,40.0,20.0,16.0
1,Школьное питание,78.0,28.0,24.0,12.0
2,Цифровые наборы,70.0,22.0,12.0,10.0
3,Зимние центры поддержки,96.0,48.0,26.0,20.0


,Ресурс,Доступный лимит
0,"Бюджет, млн руб.",88.0
1,"Трудозатраты, командо-месяцы",52.0
2,"Операционная ёмкость, слоты",42.0



## 4. Шаг 1. Прямая постановка

Базовая модель:

$$
\max E = 90x_1 + 78x_2 + 70x_3 + 96x_4
$$

при ограничениях

$$
40x_1 + 28x_2 + 22x_3 + 48x_4 \le 88,
$$

$$
20x_1 + 24x_2 + 12x_3 + 26x_4 \le 52,
$$

$$
16x_1 + 12x_2 + 10x_3 + 20x_4 \le 42,
$$

$$
0 \le x_1, x_2, x_3, x_4 \le 1.
$$

Здесь первые три неравенства отвечают за дефицитные ресурсы, а верхние границы $x_j \le 1$ означают, что нельзя профинансировать программу сверх полного масштаба.

Для `linprog` переходим к минимизации отрицательной цели:

$$
\min f = -90x_1 - 78x_2 - 70x_3 - 96x_4.
$$


In [2]:

def solve_primal(effect_vector=None, rhs_vector=None):
    effect_vector = np.array(effect if effect_vector is None else effect_vector, dtype=float)
    rhs_vector = np.array(b_resources if rhs_vector is None else rhs_vector, dtype=float)

    result = linprog(
        c=-effect_vector,
        A_ub=A_resources,
        b_ub=rhs_vector,
        bounds=bounds,
        method='highs',
    )
    if not result.success:
        raise RuntimeError(result.message)
    return result


def allocation_table(result, effect_vector=None, rhs_vector=None):
    effect_vector = np.array(effect if effect_vector is None else effect_vector, dtype=float)
    rhs_vector = np.array(b_resources if rhs_vector is None else rhs_vector, dtype=float)

    scale = result.x
    resource_use = A_resources @ scale
    slack = rhs_vector - resource_use

    allocation_df = pd.DataFrame({
        'Программа': programs,
        'Масштаб x_j': scale,
        'Процент от полного плана': 100 * scale,
        'Вклад в общественный эффект': effect_vector * scale,
        'Запас до верхней границы': 1 - scale,
    })

    resources_df = pd.DataFrame({
        'Ресурс': resources,
        'Использовано': resource_use,
        'Лимит': rhs_vector,
        'Slack': slack,
        'Binding': np.isclose(slack, 0.0, atol=1e-9),
    })
    return allocation_df, resources_df


base_result = solve_primal()
base_objective = -base_result.fun
base_allocation_df, base_resources_df = allocation_table(base_result)

print(f'Оптимальный общественный эффект: {base_objective:.6f}')
display(base_allocation_df.round(4))
display(base_resources_df.round(4))


Оптимальный общественный эффект: 226.735849


,Программа,Масштаб x_j,Процент от полного плана,Вклад в общественный эффект,Запас до верхней границы
0,Мобильные медбригады,1.0000,100.0000,90.0000,0.0000
1,Школьное питание,0.6698,66.9811,52.2453,0.3302
2,Цифровые наборы,1.0000,100.0000,70.0000,0.0000
3,Зимние центры поддержки,0.1509,15.0943,14.4906,0.8491


,Ресурс,Использовано,Лимит,Slack,Binding
0,"Бюджет, млн руб.",88.0000,88.0,-0.0000,True
1,"Трудозатраты, командо-месяцы",52.0000,52.0,-0.0000,True
2,"Операционная ёмкость, слоты",37.0566,42.0,4.9434,False



## 5. Шаг 2. Что означают `binding` и `slack`

После решения прямой задачи нас интересуют не только значения $x_j$, но и состояние ресурсов.

- Если `Slack = 0`, ресурс полностью исчерпан и реально ограничивает оптимум.
- Если `Slack > 0`, ресурс не является дефицитным в текущей точке.

В этой лабораторной важно не просто назвать активные ограничения, а сразу дать содержательную интерпретацию:

1. какой ресурс является самым узким местом;
2. есть ли ресурс, который в текущем оптимуме не лимитирует результат;
3. какие программы дошли до верхней границы `x_j = 1`.


In [3]:

binding_resources = base_resources_df.loc[base_resources_df['Binding'], 'Ресурс'].tolist()
slack_resources = base_resources_df.loc[~base_resources_df['Binding'], 'Ресурс'].tolist()
full_scale_programs = base_allocation_df.loc[
    np.isclose(base_allocation_df['Запас до верхней границы'], 0.0, atol=1e-9),
    'Программа',
].tolist()

print('Активные ресурсы:', ', '.join(binding_resources))
print('Ресурсы с запасом:', ', '.join(slack_resources))
print('Программы на полном масштабе:', ', '.join(full_scale_programs))


Активные ресурсы: Бюджет, млн руб., Трудозатраты, командо-месяцы
Ресурсы с запасом: Операционная ёмкость, слоты
Программы на полном масштабе: Мобильные медбригады, Цифровые наборы



## 6. Шаг 3. Краткая dual-модель и проверка сильной двойственности

Так как в модели есть и ресурсные ограничения, и верхние границы `x_j <= 1`, dual-задача включает два типа переменных:

- $y_1, y_2, y_3$ — теневые цены ресурсов;
- $z_1, z_2, z_3, z_4$ — оценки верхних границ программ.

Полная dual-модель:

$$
\min \Phi = 88y_1 + 52y_2 + 42y_3 + z_1 + z_2 + z_3 + z_4
$$

при ограничениях

$$
40y_1 + 20y_2 + 16y_3 + z_1 \ge 90,
$$

$$
28y_1 + 24y_2 + 12y_3 + z_2 \ge 78,
$$

$$
22y_1 + 12y_2 + 10y_3 + z_3 \ge 70,
$$

$$
48y_1 + 26y_2 + 20y_3 + z_4 \ge 96,
$$

$$
y_i \ge 0, \quad z_j \ge 0.
$$

Для экономической интерпретации нам важнее всего первые три переменные $y_i$: именно они отвечают на вопрос, сколько стоит ещё одна единица ресурса.


In [4]:

def solve_dual(effect_vector=None, rhs_vector=None):
    effect_vector = np.array(effect if effect_vector is None else effect_vector, dtype=float)
    rhs_vector = np.array(b_resources if rhs_vector is None else rhs_vector, dtype=float)

    A_full = np.vstack([A_resources, np.eye(len(programs))])
    b_full = np.concatenate([rhs_vector, np.ones(len(programs))])

    dual_result = linprog(
        c=b_full,
        A_ub=-A_full.T,
        b_ub=-effect_vector,
        bounds=[(0, None)] * len(b_full),
        method='highs',
    )
    if not dual_result.success:
        raise RuntimeError(dual_result.message)
    return dual_result, A_full, b_full


dual_result, A_full, b_full = solve_dual()
dual_objective = dual_result.fun
dual_gap = abs(base_objective - dual_objective)

resource_shadow_prices = dual_result.x[: len(resources)]
upper_bound_prices = dual_result.x[len(resources) :]

resource_shadow_df = pd.DataFrame({
    'Ресурс': resources,
    'Теневая цена y_i': resource_shadow_prices,
    'Slack ресурса': base_resources_df['Slack'].to_numpy(),
    'Комплементарность slack * y': base_resources_df['Slack'].to_numpy() * resource_shadow_prices,
})
cap_shadow_df = pd.DataFrame({
    'Программа': programs,
    'Dual-переменная z_j': upper_bound_prices,
    'Запас до границы 1 - x_j': 1 - base_result.x,
    'Комплементарность (1 - x_j) * z_j': (1 - base_result.x) * upper_bound_prices,
})

print(f'Оптимум primal: {base_objective:.6f}')
print(f'Оптимум dual:   {dual_objective:.6f}')
print(f'Абсолютный duality gap: {dual_gap:.12f}')
display(resource_shadow_df.round(6))
display(cap_shadow_df.round(6))


Оптимум primal: 226.735849
Оптимум dual:   226.735849
Абсолютный duality gap: 0.000000000000


,Ресурс,Теневая цена y_i,Slack ресурса,Комплементарность slack * y
0,"Бюджет, млн руб.",0.650943,-0.000000,-0.0
1,"Трудозатраты, командо-месяцы",2.490566,-0.000000,-0.0
2,"Операционная ёмкость, слоты",0.000000,4.943396,0.0


,Программа,Dual-переменная z_j,Запас до границы 1 - x_j,Комплементарность (1 - x_j) * z_j
0,Мобильные медбригады,14.150943,0.000000,0.0
1,Школьное питание,0.000000,0.330189,0.0
2,Цифровые наборы,25.792453,0.000000,0.0
3,Зимние центры поддержки,0.000000,0.849057,0.0



## 7. Шаг 4. Анализ чувствительности по правым частям `b`

Теперь проверяем главное свойство shadow price на практике.

Для малого изменения ресурса $\Delta b_i$ используем прогноз:

$$
\Delta E^* pprox y_i \cdot \Delta b_i.
$$

Затем решаем задачу заново и сравниваем:

1. прогноз по теневой цене;
2. фактическое изменение оптимального общественного эффекта;
3. разницу между этими двумя величинами.

Важно: если структура оптимума не изменилась, прогноз и факт обычно совпадают очень точно.


In [5]:

rhs_scenarios = [
    ('Бюджет +1', 0, 1.0),
    ('Бюджет +3', 0, 3.0),
    ('Трудозатраты +1', 1, 1.0),
    ('Трудозатраты +2', 1, 2.0),
    ('Операционная ёмкость +3', 2, 3.0),
]

rhs_rows = []
for scenario_name, resource_idx, delta in rhs_scenarios:
    rhs_new = b_resources.copy()
    rhs_new[resource_idx] += delta
    scenario_result = solve_primal(rhs_vector=rhs_new)
    predicted_delta = resource_shadow_prices[resource_idx] * delta
    actual_delta = (-scenario_result.fun) - base_objective

    rhs_rows.append({
        'Сценарий': scenario_name,
        'Ресурс': resources[resource_idx],
        'Δb': delta,
        'Прогноз по y_i * Δb': predicted_delta,
        'Факт после пересчёта': actual_delta,
        'Разница': actual_delta - predicted_delta,
        'Новый план': '; '.join(f'{name}={value:.3f}' for name, value in zip(programs, scenario_result.x)),
    })

rhs_sensitivity_df = pd.DataFrame(rhs_rows)
display(rhs_sensitivity_df.round(6))


,Сценарий,Ресурс,Δb,Прогноз по y_i * Δb,Факт после пересчёта,Разница,Новый план
0,Бюджет +1,"Бюджет, млн руб.",1.0,0.650943,0.650943,-0.0,Мобильные медбригады=1.000; Школьное питание=0...
1,Бюджет +3,"Бюджет, млн руб.",3.0,1.952830,1.952830,-0.0,Мобильные медбригады=1.000; Школьное питание=0...
2,Трудозатраты +1,"Трудозатраты, командо-месяцы",1.0,2.490566,2.490566,0.0,Мобильные медбригады=1.000; Школьное питание=0...
3,Трудозатраты +2,"Трудозатраты, командо-месяцы",2.0,4.981132,4.981132,-0.0,Мобильные медбригады=1.000; Школьное питание=0...
4,Операционная ёмкость +3,"Операционная ёмкость, слоты",3.0,0.000000,-0.000000,-0.0,Мобильные медбригады=1.000; Школьное питание=0...



## 8. Шаг 5. Анализ чувствительности по коэффициентам цели `c`

Здесь мы меняем уже не ресурсы, а саму ценность программ.

Проверим два типа сценариев:

1. **малое изменение**, при котором структура плана может сохраниться;
2. **более сильное изменение**, которое уже способно переключить оптимум на другой набор программ.


In [6]:

objective_scenarios = [
    ('Школьное питание: +6 к эффекту', 1, 6.0),
    ('Зимние центры: -30 к эффекту', 3, -30.0),
]

objective_rows = []
objective_results = {}

for scenario_name, program_idx, delta in objective_scenarios:
    effect_new = effect.copy()
    effect_new[program_idx] += delta
    scenario_result = solve_primal(effect_vector=effect_new)
    objective_results[scenario_name] = scenario_result

    objective_rows.append({
        'Сценарий': scenario_name,
        'Новый коэффициент': effect_new[program_idx],
        'Новый оптимум': -scenario_result.fun,
        'План изменился': not np.allclose(scenario_result.x, base_result.x),
        'Новый план': '; '.join(f'{name}={value:.3f}' for name, value in zip(programs, scenario_result.x)),
    })

objective_sensitivity_df = pd.DataFrame(objective_rows)
display(objective_sensitivity_df)

compare_df = pd.DataFrame({
    'Базовый план': base_result.x,
    'Сценарий: зимние центры -30': objective_results['Зимние центры: -30 к эффекту'].x,
}, index=programs)
display(compare_df.round(4))


,Сценарий,Новый коэффициент,Новый оптимум,План изменился,Новый план
0,Школьное питание: +6 к эффекту,84.0,230.754717,False,Мобильные медбригады=1.000; Школьное питание=0...
1,Зимние центры: -30 к эффекту,66.0,225.000000,True,Мобильные медбригады=1.000; Школьное питание=0...


,Базовый план,Сценарий: зимние центры -30
Мобильные медбригады,1.0000,1.0000
Школьное питание,0.6698,0.8333
Цифровые наборы,1.0000,1.0000
Зимние центры поддержки,0.1509,0.0000



## 9. Что обязательно проговорить в выводе

1. Какие два ресурса оказались действительно дефицитными?
2. Почему ресурс с положительным запасом имеет нулевую или близкую к нулю теневую цену?
3. Какой ресурс даёт наибольший прирост общественного эффекта на дополнительную единицу лимита?
4. Какие программы упираются в верхнюю границу полного масштаба?
5. Когда изменение коэффициентов цели меняет только значение оптимума, а когда меняет и сам план?

## 10. Что должно быть в отчёте

1. Полная таблица программ и ресурсов.
2. Прямая постановка модели.
3. `c`, `A_ub`, `b_ub`, `bounds`.
4. Решение прямой задачи и интерпретация плана.
5. Таблица `binding/slack`.
6. Краткая запись dual-модели.
7. Численная проверка сильной двойственности.
8. Таблица shadow prices ресурсов.
9. Сценарии по изменениям `b` и сравнение прогноза с фактом.
10. Сценарии по изменениям `c` и вывод о смене или сохранении структуры оптимума.
